In [ ]:
from pathlib import Path
import sys
from dotenv import load_dotenv

PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

load_dotenv(PROJECT_ROOT / ".env")
print(PROJECT_ROOT)


from src.config import parent_store_collection
from src.generation.generator import generate_answer
from src.retrieval.retriever import search_vector_db
from src.caching.semantic_cache import get_semantic_cache, set_semantic_cache
from langchain_groq import ChatGroq
from src.generation.generator import llm
from src.config import OPENAI_MODEL_GROQ, TEMPERATURE, MAX_TOKENS, GROQ_API_KEY
from langchain_community.tools import DuckDuckGoSearchRun
import re

In [ ]:
llm_evaluator_for_retrieval = ChatGroq(
    model=OPENAI_MODEL_GROQ,
    api_key=GROQ_API_KEY,
    temperature=0,
    max_tokens=MAX_TOKENS,
)

In [ ]:
def llm_router(dict):
    prompt = f"""
    You are a strict document evaluator for a RAG system.
    You're task is to rate each query and doc pair and tell if it is "correct", "incorrect", or "ambiguous".
    "correct" means the document is highly relevant to the query.
    "incorrect" means the document is not relevant to the query.
    "ambiguous" means the document is partially relevant or you are unsure.
    Answer only "correct", "incorrect", or "ambiguous".

Query : {dict['query']}
Doc : {dict['doc']}

Answer:
    """
    return llm_evaluator_for_retrieval.invoke(prompt).content.strip()

In [ ]:
def get_documents_for_evaluation(query, namespace: str = "default_namespace"):
    retrieved_chunks = search_vector_db(
        namespace=namespace, 
        query=query, 
        top_k=3
    )
    
    parent_ids = {chunk.get("parent_id") for chunk in retrieved_chunks if chunk.get("parent_id")}
    
    documents = []
    if parent_ids:
        parent_docs = list(parent_store_collection.find({
            "parent_id": {"$in": list(parent_ids)},
            "namespace": namespace
        }))
        for doc in parent_docs:
            documents.append(doc["text"])
    else:
        for chunk in retrieved_chunks:
            documents.append(chunk["chunk_text"])
            
    return documents

In [ ]:
query = "Tell me about Cross-Modal Knowledge Graph"
documents = get_documents_for_evaluation(query)
print(f"Retrieved {len(documents)} documents.")

In [ ]:
def clean_and_split(text: str):
    text = re.sub(r'-\s*\n\s*', '', text)
    text = text.replace('\n', ' ')
    text = re.sub(r'\s+', ' ', text).strip()
    sentences = re.split(r'(?<=[.!?])\s+', text)
    return [s.strip() for s in sentences if len(s.strip()) > 20]

In [ ]:
def web_search(query):
    print("\n=> Triggering web search fallback for query:", query)
    search = DuckDuckGoSearchRun()
    try:
        results = search.invoke(query)
        return f"[Web Search Results]: {results}"
    except Exception as e:
        print(f"Web search failed: {e}")
        return ""

In [ ]:
final_context_strips = []
needs_web_search = False

for i, doc in enumerate(documents):
    eval_result = llm_router({"query": query, "doc": doc})
    print(f"Document {i+1} Evaluation: {eval_result}")
    
    if eval_result.lower() == "correct":
        strips = clean_and_split(doc)
        for strip in strips:
            strip_eval = llm_router({"query": query, "doc": strip})
            if strip_eval.lower() in ["correct", "ambiguous", "good"]:
                final_context_strips.append(strip)
    elif eval_result.lower() == "ambiguous":
        needs_web_search = True
        strips = clean_and_split(doc)
        for strip in strips:
            strip_eval = llm_router({"query": query, "doc": strip})
            if strip_eval.lower() in ["correct", "ambiguous", "good"]:
                final_context_strips.append(strip)
    else:
        needs_web_search = True

if needs_web_search or len(final_context_strips) == 0:
    external_knowledge = web_search(query)
    if external_knowledge:
        final_context_strips.append(external_knowledge)

final_context = "\n".join(final_context_strips)
print("\n--- Final Refined Context ---")
print(final_context)

In [ ]:

messages = [
    ("system", "You are an assistant. Answer the query based on the context."),
    ("human", f"Context:\n{final_context}\n\n---\nQuestion: {query}")
]
response = llm.invoke(messages)
print("\n--- Final Generated Answer ---")
print(response.content)